# 🔀 Multi-Query Retrieval

A single question is a single point in embedding space. If the user's phrasing happens to sit far
from how the documents are worded, relevant chunks are simply missed — and you never find out,
because retrieval fails silently.

**Multi-Query** fixes this by asking an LLM to rewrite the question several ways, retrieving for
each variation, and merging the results. More phrasings means more chances to land near the right
chunk.

## Learning Objectives
1. **The single-phrasing problem** — why one query vector is a fragile way to search
2. **Query generation** — produce reliable variations with structured output, not string splitting
3. **Parallel retrieval** — use `retriever.map()` to search once per variation
4. **Merging results** — deduplicate correctly, and see why the obvious approach silently fails
5. **The full chain** — wire multi-query retrieval into an end-to-end RAG pipeline

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Source documents in `04_Retrieval_and_RAG/shared_data/`
- Familiarity with embeddings and vector stores (see `01_Introduction_to_RAG/`)

---
## 🧠 Part 1: Why One Query Is Not Enough

Dense retrieval embeds the question and returns the nearest chunks. That makes the result entirely
dependent on **how the question happens to be worded**.

Consider three ways of asking the same thing:

| Phrasing | Vocabulary it embeds near |
|---|---|
| *"What is LangSmith?"* | definitions, product overviews |
| *"How do I debug a failing LLM chain?"* | debugging, tracing, errors |
| *"Why do LLM apps break in production?"* | reliability, monitoring, incidents |

All three should surface the same LangSmith documentation, but each lands in a different
neighbourhood of the vector space. A single query retrieves only one neighbourhood.

### Key Concepts:
- **Query variation**: an LLM-generated rephrasing that targets a different facet of the intent.
- **Fan-out retrieval**: running the retriever once per variation, in parallel.
- **Union merge**: combining the result lists into one deduplicated set.

> **Key Insight**: multi-query does not make the retriever smarter. It **widens the net**, trading
> extra LLM and search calls for better recall.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

from pydantic import BaseModel, Field

# LangChain core
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Integrations
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

from dotenv import load_dotenv
from operator import itemgetter

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT`. The SDK checks the
> `LANGSMITH_` prefix **first**, so a `LANGSMITH_PROJECT` already set in `.env` would silently win
> and these traces would land in that project instead of this one.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "Multi-Query"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the Models

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: LLM (writes variations) + embeddings (search)
# ============================================================================
llm = get_experientiallabs_llm()

# Pinned explicitly: a bare OpenAIEmbeddings() still defaults to legacy ada-002.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"🤖 LLM:        {llm.model_name}")
print(f"🔢 Embeddings: {embeddings.model}")

---
## 📚 Part 3: Build the Knowledge Base

Two LangChain blog posts, chunked at 400 tokens with 60 tokens of overlap. The overlap matters:
without it an idea split across a boundary becomes unretrievable from either side.

In [ ]:
# ============================================================================
# KNOWLEDGE BASE: Load, split, and index
# ============================================================================
loaders = [
    TextLoader("../../shared_data/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../../shared_data/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400,    # tokens, not characters
    chunk_overlap=60,  # keeps ideas intact across boundaries
)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print(f"📄 Loaded {len(docs)} documents → {len(splits)} chunks → indexed in Chroma")

---
## 🔍 Part 4: Baseline — Retrieval With a Single Query

Establish what one phrasing retrieves, so the multi-query result has something to be compared
against. Note which chunks appear here; several will be missing that the variations later surface.

In [ ]:
# ============================================================================
# BASELINE: One query, one set of results
# ============================================================================
question = "What is LangSmith, and why do we need it?"

baseline_docs = retriever.invoke(question)

print(f"🔍 Single-query baseline — {len(baseline_docs)} chunks\n")
for i, doc in enumerate(baseline_docs, 1):
    print(f"[{i}] {doc.page_content[:110].strip()}...")

---
## ✍️ Part 5: Generate Query Variations

### 5.1 Why Structured Output, Not `split("\n")`

The original version of this notebook parsed the LLM's response with
`(lambda x: x.split("\n"))`. That is fragile: it assumes the model returns exactly N bare lines.
In practice models add headers, numbering, blank lines, or a preamble like *"Here are three
variations:"* — and every one of those becomes a "query" that gets sent to the retriever.

Worse, it is **intermittent**. The same chain returns clean output on one run and markdown on the
next, so the bug appears and disappears between runs.

`with_structured_output()` binds a Pydantic schema to the model, so the provider returns validated
JSON. You get a guaranteed `list[str]` regardless of how the model feels like formatting prose.

> **Rule of thumb**: any time you are about to `.split()` or regex an LLM response to get a list,
> reach for `with_structured_output()` instead.

In [ ]:
# ============================================================================
# QUERY GENERATION: Schema-validated variations
# ============================================================================
class QueryVariations(BaseModel):
    """Alternative phrasings of the user's question."""

    queries: list[str] = Field(
        description="Three alternative phrasings, each targeting a different facet of the intent"
    )


template = """You are helping a vector search engine find relevant documents.

The user asked: "{question}"

Write three alternative phrasings of this question. Each should target a different
facet of the user's intent and use different vocabulary, so that documents are found
even when they do not share keywords with the original question."""

prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | llm.with_structured_output(QueryVariations)
    | (lambda x: x.queries)
)

print("✅ Query generation chain ready")

### 5.2 Look at the Generated Queries

This is the cell worth reading carefully. Notice how each variation reaches for **different
vocabulary** — that vocabulary spread is the entire mechanism of the technique.

In [ ]:
# ============================================================================
# INSPECTION: What variations did the LLM actually produce?
# ============================================================================
variations = generate_queries.invoke({"question": question})

print(f"❓ Original: {question}\n")
print(f"📋 {len(variations)} generated variations:")
for i, q in enumerate(variations, 1):
    print(f"   {i}. {q}")

---
## 🗂️ Part 6: Retrieve for Every Variation

`retriever.map()` lifts the retriever so it accepts a **list** of queries and returns a list of
result lists — one per variation, run in parallel. The result is a `list[list[Document]]`.

In [ ]:
# ============================================================================
# FAN-OUT RETRIEVAL: One result list per variation
# ============================================================================
per_query_docs = retriever.map().invoke(variations)

print(f"🔍 Retrieved {len(per_query_docs)} result lists\n")
for q, hits in zip(variations, per_query_docs):
    print(f"❓ {q}")
    print(f"   📊 {len(hits)} chunks: {[h.page_content[:40].strip() + '...' for h in hits[:2]]}\n")

total = sum(len(h) for h in per_query_docs)
print(f"📊 {total} chunks retrieved in total (before deduplication)")

---
## 🔗 Part 7: Merge the Results

### 7.1 A Bug Worth Studying

The original `get_unique_union()` in this notebook was silently broken:

```python
if hasattr(flattened_docs[0], 'id'):
    unique_docs = list(set(doc.id for doc in flattened_docs))   # returns IDs, not documents!
else:
    unique_docs = list(set(str(doc) for doc in flattened_docs))  # returns strings, not documents!
```

Every `Document` has an `.id` attribute, so the first branch always ran and the function returned a
list of **bare identifiers** — UUID strings, or `None` values on stores that leave `.id` unset. The
original notebook's output shows `len(docs) == 1`, because a list of `None`s collapses to `{None}`.

The failure is invisible: those identifiers were passed to the prompt as `{context}`, the LLM saw
no usable text, and answered from its own training knowledge instead. **The final answer looked
completely reasonable while retrieval contributed nothing.**

> **⚠️ Lesson**: a RAG pipeline that returns a fluent answer is not evidence that retrieval worked.
> Always inspect what actually lands in `{context}`.

In [ ]:
# ============================================================================
# MERGE: Deduplicate on content, returning real Documents
# ============================================================================
def get_unique_union(documents: list[list[Document]]) -> list[Document]:
    """Flatten result lists and drop duplicates, preserving Document objects.

    Deduplicates on page_content: the same chunk retrieved by two different
    variations is one result, not two.
    """
    seen: set[str] = set()
    unique: list[Document] = []

    for sublist in documents:
        for doc in sublist:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique.append(doc)

    return unique


unique_docs = get_unique_union(per_query_docs)

print(f"📊 {total} retrieved → {len(unique_docs)} unique chunks")
print(f"📊 Single-query baseline found {len(baseline_docs)}")
print(f"✅ Multi-query surfaced {len(unique_docs) - len(baseline_docs)} additional chunk(s)")

### 7.2 What Did the Extra Queries Actually Add?

Comparing against the Part 4 baseline shows whether the extra LLM and retrieval calls bought
anything. If the variations return exactly what the original query returned, multi-query is pure
overhead for this corpus — worth knowing rather than assuming.

In [ ]:
# ============================================================================
# COMPARISON: Which chunks did multi-query add over the baseline?
# ============================================================================
baseline_content = {d.page_content for d in baseline_docs}
new_docs = [d for d in unique_docs if d.page_content not in baseline_content]

print(f"📊 Chunks unique to multi-query: {len(new_docs)}\n")
for i, doc in enumerate(new_docs, 1):
    print(f"🆕 [{i}] {doc.page_content[:140].strip()}...\n")

if not new_docs:
    print("⚠️  No new chunks. On a small, well-matched corpus the variations often")
    print("   converge on the same results — the cost is real, the benefit is not.")

---
## 🎯 Part 8: The Complete RAG Chain

Now assemble everything: generate variations → retrieve for each → merge → answer. The retrieval
chain becomes the `context` input to a normal RAG prompt.

In [ ]:
# ============================================================================
# RETRIEVAL CHAIN: variations -> parallel retrieval -> unique union
# ============================================================================
retrieval_chain = generate_queries | retriever.map() | get_unique_union

print("✅ Multi-query retrieval chain ready")

In [ ]:
# ============================================================================
# RAG CHAIN: Retrieval feeds the answer prompt
# ============================================================================
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

answer = final_rag_chain.invoke({"question": question})
print(answer)

### 8.1 Verify the Context Is Real

Given the bug in Part 7.1, it is worth confirming that `{context}` genuinely contains document
text. This is the check that would have caught the original failure.

In [ ]:
# ============================================================================
# SANITY CHECK: Confirm retrieval actually reaches the prompt
# ============================================================================
context_docs = retrieval_chain.invoke({"question": question})

print(f"📊 Context contains {len(context_docs)} items")
print(f"📋 Type of each item: {type(context_docs[0]).__name__}")

assert all(isinstance(d, Document) for d in context_docs), "Context must hold Document objects!"
assert all(d.page_content.strip() for d in context_docs), "Documents must carry real text!"

print("✅ Context holds real Document objects with real text")
print(f"\n📄 First chunk: {context_docs[0].page_content[:180].strip()}...")

---
## 📝 Summary

### 1. The Problem
- A single query is a single point in embedding space. If the user's wording sits far from the
  document's wording, relevant chunks are **silently** missed.

### 2. How Multi-Query Works
- An LLM writes several phrasings of the question, each is retrieved for in parallel via
  `retriever.map()`, and the result lists are merged into a deduplicated union.
- The mechanism is **vocabulary spread** — Part 5.2 showed each variation reaching for different terms.

### 3. Generate Lists With Structured Output
- `with_structured_output()` + a Pydantic model guarantees a `list[str]`.
- String-splitting an LLM response is intermittently wrong: it works until the model adds a header
  or a blank line, and then feeds junk straight into the retriever.

### 4. Merging Is Where Bugs Hide
- The original `get_unique_union()` returned **document IDs instead of documents**, so `{context}`
  held opaque identifiers and the LLM answered from memory.
- The pipeline still produced a confident, plausible answer. **A good answer is not evidence that
  retrieval worked** — check what lands in the context (Part 8.1).

### 5. Costs and When It Helps
- Costs one extra LLM call plus N retrievals per question.
- Wins on short, ambiguous, or jargon-mismatched queries. On a small, homogeneous corpus the
  variations often converge on the same chunks — compare against a baseline (Part 7.2) instead of
  assuming a benefit.

### Next Steps
- Inspect these runs in LangSmith under the **Multi-Query** project to see the fan-out.
- Continue to `b. RAG_Fusion.ipynb`, which retrieves the same way but **fuses the rankings** with
  Reciprocal Rank Fusion instead of taking a flat union — so chunks found by several variations
  rise to the top.
- Compare with `c. Step_Back_Prompting` and `d. HyDE`: all rewrite the query, each in a different
  direction.